In [ ]:
from pathlib import Path
import pickle 
import matplotlib.pyplot as plt
import numpy as np 

In [ ]:
# load registration quality metrics (SSIM, registration error, Dice coefficient)
# load glomerulus metrics for each model 

glomerulus_metrics_path = r"" # path to glomerulosclerosis model results 

In [ ]:
SSIM_path =  r"" # path to SSIM results
dice_path = r"" # path to DICE results
error_path = r"" # path to registration error results

In [ ]:
with open(SSIM_path, 'rb') as f:
    SSIM = pickle.load(f)

with open(dice_path, 'rb') as f:
    dice = pickle.load(f)

with open(error_path, 'rb') as f:
    registration_error = pickle.load(f)

In [ ]:
metric = 'auc' 
model = 'mlp'

model_metrics_by_sample_path = [f for f in Path(glomerulus_metrics_path).glob('*.pickle') if metric in str(f) and model in str(f)][0]

In [ ]:
metric_converted = {'f1': 'F1', 'auc': 'AUC-ROC'}

In [ ]:
model_converted = {'mlp': 'MLP', 'rf': 'Random Forest', 'lr': 'Logistic Regression'}

In [ ]:
with open(model_metrics_by_sample_path, 'rb') as f: 
    model_metrics_by_sample = pickle.load(f)

In [ ]:
sample_path = {} # "sample name: sample identifier used in path"

In [ ]:
fold_sample = {0: 'HKT CTRL 1', 1: 'HKT Infection', 2: 'HKT ANCA'}

In [ ]:
sample_metrics_ssim = {}

for fold, sample in fold_sample.items():
    sample_metrics_ssim[sample] = [np.mean(SSIM[sample]), model_metrics_by_sample[fold]]

In [ ]:
sample_metrics_dice = {}

for fold, sample in fold_sample.items():
    sample_metrics_dice[sample] = [np.mean(list(dice[sample_path[sample]].values())), model_metrics_by_sample[fold]]

In [ ]:
plt.rcParams['font.size'] = 20

sorted_sample_metrics_ssim = sorted(sample_metrics_ssim.items(), key=lambda item: item[1][0])
sorted_sample_metrics_dice = sorted(sample_metrics_dice.items(), key=lambda item: item[1][0])

x1 = [point[1][0] for point in sorted_sample_metrics_ssim]
y1 = [point[1][1] for point in sorted_sample_metrics_ssim]

x2 = [point[1][0] for point in sorted_sample_metrics_dice]
y2 = [point[1][1] for point in sorted_sample_metrics_dice]

plt.plot(x1, y1, marker="o", label='SSIM')
plt.plot(x2, y2, marker="o", label='Dice coefficient')

plt.xlabel("Registration Quality")
plt.ylabel("Model Performance")
plt.title(model_converted[model] + ' ' + metric_converted[metric])
plt.legend()
plt.tight_layout()
plt.ylim(0, 1)
plt.show()
